In [1]:
# ==========================================
#   IMPORT LIBRARIES & SETUP
# ==========================================
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'   # implicit recommends this for ALS

%load_ext autoreload
%autoreload 2

from metrics import evaluate
import pandas as pd 
import numpy as np
from scipy.sparse import csr_matrix
import implicit
from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import CosineRecommender
import itertools
import time

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
articles=pd.read_parquet('data/articles_processed.parquet')
customers=pd.read_parquet('data/customers_processed.parquet')
transactions=pd.read_parquet('data/transactions_processed.parquet')


In [3]:
#=================================================================================
#                        MAPPING
#=================================================================================

unique_customers = customers['customer_id'].unique()
unique_articles = articles['article_id'].unique()

# Mappings
customer2idx = {c: i for i, c in enumerate(unique_customers)}
article2idx = {a: i for i, a in enumerate(unique_articles)}

# Reverse Mappings 
idx2customer = {i: c for c, i in customer2idx.items()}
idx2article = {i: a for a, i in article2idx.items()}

# Mapping transactions-ի վրա
transactions['user_idx'] = transactions['customer_id'].map(customer2idx)
transactions['item_idx'] = transactions['article_id'].map(article2idx)

# Check: no unmapped IDs
print(transactions['user_idx'].isna().sum(), transactions['item_idx'].isna().sum())

0 0


In [4]:
#================================================================================
#                  TRAIN-VALIDATION-TEST SPLIT
#================================================================================
last_date = transactions['t_dat'].max()
test_start = last_date - pd.Timedelta(days=6)    # last week
val_start = test_start - pd.Timedelta(days=7)    # the week before last

train = transactions[transactions['t_dat'] < val_start]
val = transactions[(transactions['t_dat'] >= val_start) & (transactions['t_dat'] < test_start)]
test = transactions[transactions['t_dat'] >= test_start]

train = pd.concat([train, val])

In [5]:
#=================================================================================
#                              CSR Matrix
#=================================================================================

train_user_item = train.groupby(['user_idx', 'item_idx']).size().reset_index(name='weight')

rows = train_user_item['user_idx'].values
cols = train_user_item['item_idx'].values
weights = train_user_item['weight'].values.astype(np.float32)

n_users = len(customer2idx)
n_items = len(article2idx)


user_item_matrix = csr_matrix((weights, (rows, cols)), shape=(n_users, n_items))


In [6]:

recent_popular = train[train['t_dat'] >= train['t_dat'].max() - pd.Timedelta(days=6)]
fallback_top12 = recent_popular.groupby('article_id').size().nlargest(12).index.tolist()

def predict_implicit_model(model, val_df, k=12):
    
    actuals_dict = val_df.groupby('customer_id')['article_id'].unique().apply(set).to_dict()
    val_customers = list(actuals_dict.keys())

    user_idxs = np.array([customer2idx[c_id] for c_id in val_customers])

    has_history = np.asarray(user_item_matrix[user_idxs].getnnz(axis=1)) > 0

    ids, scores = model.recommend(
        user_idxs,
        user_item_matrix[user_idxs],
        N=k,
        filter_already_liked_items=False
    )

    predictions = {}
    for i, c_id in enumerate(val_customers):
        if has_history[i]:
            recs = [idx2article[int(idx)] for idx in ids[i] if idx >= 0]
        else:
            recs = []  # cold customer
        fill = [p for p in fallback_top12 if p not in recs]
        predictions[c_id] = (recs + fill)[:k]

    return predictions, actuals_dict